In [17]:
import csv, json

PATH = "/appointments.csv"

def parse_row(row):
    row = dict(row)
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row

accepted, rejected = [], []

with open(PATH, newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    for row in reader:
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

print("Rows read:", len(accepted) + len(rejected))
print("Accepted:", len(accepted), "| Rejected:", len(rejected))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988 | Rejected: 32


In [18]:
import pandas as pd, numpy as np

df = pd.read_csv("/appointments.csv")
print(df.shape)
df.info()
print(df.isnull().sum())
print("duplicates:", df.duplicated().sum())
print("raw channel spellings:", df["booking_channel"].nunique())

df = df.drop_duplicates()                                               # (a)
print("after (a):", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()   # (b)
print("channels now:", df["booking_channel"].unique())

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")      # (c)
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("after (c):", len(df))

df = df.dropna(subset=["appt_date", "status"])                          # (d)
print("after (d):", len(df))


(1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on      

In [19]:
# (a) impossible ages -> missing
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan
print("missing ages:", df["age"].isna().sum())

# (b) fill gaps with medians
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
print("median age:", df["age"].median(), "| median distance:", df["distance_km"].median())

# (c) cap outliers with IQR fences
for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    capped = ((df[col] < low) | (df[col] > high)).sum()
    df[col] = np.clip(df[col], low, high)
    print(col, "| Q1", q1, "| Q3", q3, "| upper fence", high, "| capped", capped)

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("max fee:", df["fee_inr"].max())


missing ages: 17
median age: 48.0 | median distance: 8.25
distance_km | Q1 4.625 | Q3 13.0 | upper fence 25.5625 | capped 26
fee_inr | Q1 500.0 | Q3 787.5 | upper fence 1218.75 | capped 3
max fee: 1219


In [20]:
# Features
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("no-shows:", df["no_show"].sum(), "of", len(df), "=", round(100 * df["no_show"].mean(), 1), "%")

# NumPy summary
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
print("means:", X.mean(axis=0).round(2))

# (b) lead_days > 14
mask = df["lead_days"].to_numpy() > 14
print("lead_days > 14:", mask.sum(), "rows =", round(100 * mask.mean(), 1), "%")

# (c) z-scores of distance_km
x = df["distance_km"].to_numpy()
z = (x - np.mean(x)) / np.std(x)
print("|z| > 2:", (np.abs(z) > 2).sum())


lead_days range: 1 to 45
no-shows: 218 of 990 = 22.0 %
means: [ 48.17   9.5  635.71  11.82]
lead_days > 14: 296 rows = 29.9 %
|z| > 2: 53


In [21]:
import requests

clinics = pd.read_csv("/clinics.csv")
URL = "https://api.open-meteo.com/v1/forecast"

def get_temp(lat, lon):
    params = {"latitude": lat, "longitude": lon,
              "hourly": "temperature_2m", "forecast_days": 1}
    r = requests.get(URL, params=params, timeout=10)
    if r.status_code != 200:
        raise ValueError(f"status {r.status_code}")
    return r.json()["hourly"]["temperature_2m"][0]   # midnight-UTC reading

temps = []
for _, row in clinics.iterrows():
    temp, reason = None, ""
    for attempt in (1, 2):                            # one retry
        try:
            temp = get_temp(row["latitude"], row["longitude"])
            break
        except (requests.exceptions.RequestException, ValueError, KeyError, IndexError) as e:
            reason = str(e)
    if temp is None:
        print("FAILED:", row["clinic_name"], "-", reason)
    temps.append(temp)

clinics["temp_c"] = temps
print(clinics[["clinic_id", "clinic_name", "temp_c"]])
clinics.to_csv("clinics_enriched.csv", index=False)


  clinic_id               clinic_name  temp_c
0       C01     Kothrud Family Clinic    16.0
1       C02      Hadapsar Care Centre    17.3
2       C03    Sitabuldi Health Point    20.8
3       C04      Malviya Nagar Clinic    15.8
4       C05     Vaishali Nagar Clinic    15.8
5       C06  Edappally Medical Centre    25.8
6       C07        Vijay Nagar Clinic    17.4
7       C08    Gomti Nagar Care Point    17.7


In [22]:
cols = ["appointment_id","patient_id","clinic_id","booked_on","appt_date","age","gender",
        "distance_km","fee_inr","booking_channel","visit_type","reminder_sent",
        "prior_no_shows","status","lead_days","no_show"]

# sort and format dates as YYYY-MM-DD text
df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
out = df[cols]

# checks before exporting
print("unique ids:", out["appointment_id"].nunique(), "| is_unique:", out["appointment_id"].is_unique)
print("missing values:", out.isnull().sum().sum())

out.to_csv("appointments_clean.csv", index=False)
print("shape:", out.shape)

# SQL load script: clinics first (foreign key), then appointments
lines = []
for _, r in clinics.iterrows():
    temp = "NULL" if pd.isna(r["temp_c"]) else r["temp_c"]
    lines.append(f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}','{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{temp});")

for _, r in out.iterrows():
    lines.append(f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});")

with open("load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")

print("INSERT statements:", len(lines))
print(lines[0]); print(lines[8])


unique ids: 990 | is_unique: True
missing values: 0
shape: (990, 16)
INSERT statements: 998
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.0);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
